# Fall 2025 DS701 Grade Calculation



In [ ]:
import pandas as pd

df = pd.read_csv('DS701_Fall_2025_grades.csv')


In [ ]:
print(df.columns)


## Lateness and Penalty Calculations

In [ ]:
import re

def parse_lateness_to_hours(lateness_str):
    """Convert H:M:S lateness string to total hours as float."""
    if pd.isna(lateness_str) or lateness_str == '00:00:00':
        return 0.0
    # Parse H:M:S format
    parts = str(lateness_str).split(':')
    if len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2])
        return hours + minutes / 60 + seconds / 3600
    return 0.0

def apply_late_penalty(score, lateness_hours):
    """Apply 10% penalty if lateness is > 0 and < 48 hours."""
    if pd.isna(score):
        return 0.0
    if lateness_hours > 0 and lateness_hours < 48:
        return score * 0.9
    return score


## Find and Assign Categories to Assignments

In [ ]:
# Get all unique assignment names by finding columns that don't have suffixes
assignment_columns = []
for col in df.columns:
    if ' - Max Points' in col:
        assignment_name = col.replace(' - Max Points', '')
        assignment_columns.append(assignment_name)

print("Found assignments:")
for a in assignment_columns:
    print(f"  - {a}")


In [ ]:
# Define assignment categories
midterm_challenge_assignments = [
    'Midterm Challenge -- Sports',
    'Midterm Challenge -- Health'
]

final_project_assignments = [
    'Project Final Submission, Repo, Presentation and Report',
    'Final Project Personal Contributions'
]

# In-class activities = specific assignments plus any with 'in-class activity' in name
in_class_specific = [
    'AI Use Reflection',
    'Intro Survey',
    'Spark Project Pitches and Preference Form',
    'Spark Midterm Survey'
]

in_class_activities = [
    a for a in assignment_columns 
    if 'in-class activity' in a.lower() or a in in_class_specific
]

# Homework = everything except midterm challenge, final project, and in-class activities
homework_assignments = [
    a for a in assignment_columns 
    if a not in midterm_challenge_assignments 
    and a not in final_project_assignments
    and a not in in_class_activities
]

print("Homework Assignments:")
for a in homework_assignments:
    print(f"  - {a}")
    
print("\nIn-Class Activities:")
for a in in_class_activities:
    print(f"  - {a}")
    
print("\nMidterm Challenge Assignments:")
for a in midterm_challenge_assignments:
    print(f"  - {a}")
    
print("\nFinal Project Assignments:")
for a in final_project_assignments:
    print(f"  - {a}")


In [ ]:
# Calculate scores with late penalties applied
def get_adjusted_score(row, assignment_name):
    """Get the score for an assignment with late penalty applied if applicable."""
    score_col = assignment_name
    lateness_col = f"{assignment_name} - Lateness (H:M:S)"
    
    if score_col not in df.columns:
        return 0.0
    
    score = row[score_col]
    if pd.isna(score):
        return 0.0
    
    # Ignore lateness for 'Individual Reflection' and 'Personal Contributions' assignments
    if 'Individual Reflection' in assignment_name:
        return score
    if 'Personal Contributions' in assignment_name:
        return score
    
    # Get lateness if available
    if lateness_col in df.columns:
        lateness_hours = parse_lateness_to_hours(row[lateness_col])
        return apply_late_penalty(score, lateness_hours)
    else:
        return score

# Calculate Total Homework Score (with late penalties)
def calculate_homework_total(row):
    total = 0.0
    for assignment in homework_assignments:
        total += get_adjusted_score(row, assignment)
    return total

df['Total Homework Score'] = df.apply(calculate_homework_total, axis=1)

# Calculate Midterm Challenge Score (max of Sports and Health, with penalties)
def calculate_midterm_challenge(row):
    sports_score = get_adjusted_score(row, 'Midterm Challenge -- Sports')
    health_score = get_adjusted_score(row, 'Midterm Challenge -- Health')
    return max(sports_score, health_score)

df['Midterm Challenge Score'] = df.apply(calculate_midterm_challenge, axis=1)

# Calculate Final Project Total (sum of Final Project + Personal Contributions, with penalties)
def calculate_final_project_total(row):
    final_project_score = get_adjusted_score(row, 'Project Final Submission, Repo, Presentation and Report')
    personal_contributions_score = get_adjusted_score(row, 'Final Project Personal Contributions')
    return final_project_score + personal_contributions_score

df['Final Project Total Score'] = df.apply(calculate_final_project_total, axis=1)

# Calculate In-Class Activities Total (with penalties)
def calculate_in_class_total(row):
    total = 0.0
    for assignment in in_class_activities:
        total += get_adjusted_score(row, assignment)
    return total

df['In-Class Activities Score'] = df.apply(calculate_in_class_total, axis=1)

print("New columns added:")
print("  - Total Homework Score")
print("  - In-Class Activities Score")
print("  - Midterm Challenge Score")
print("  - Final Project Total Score")


In [ ]:
# Display the new columns with student info
result_df = df[['Name', 'SID', 'Email', 'Total Homework Score', 'In-Class Activities Score', 'Midterm Challenge Score', 'Final Project Total Score']]
print(f"Total students: {len(result_df)}")
print(f"\nHomework assignments count: {len(homework_assignments)}")
print(f"In-class activities count: {len(in_class_activities)}")

# Calculate max possible homework score (sum of max points for all homework assignments)
max_homework_points = 0
for assignment in homework_assignments:
    max_col = f"{assignment} - Max Points"
    if max_col in df.columns:
        max_val = df[max_col].max()
        if pd.notna(max_val):
            max_homework_points += max_val

# Calculate max possible in-class activities score
max_in_class_points = 0
for assignment in in_class_activities:
    max_col = f"{assignment} - Max Points"
    if max_col in df.columns:
        max_val = df[max_col].max()
        if pd.notna(max_val):
            max_in_class_points += max_val
            
print(f"\nMax possible homework points: {max_homework_points}")
print(f"Max possible in-class activities points: {max_in_class_points}")
print(f"Max Midterm Challenge points: {df['Midterm Challenge -- Sports - Max Points'].max()} (per option)")
print(f"Max Final Project points: {df['Project Final Submission, Repo, Presentation and Report - Max Points'].max()} + {df['Final Project Personal Contributions - Max Points'].max()} = {df['Project Final Submission, Repo, Presentation and Report - Max Points'].max() + df['Final Project Personal Contributions - Max Points'].max()}")

result_df.head(20)


In [ ]:
# Show examples of late penalty applications
# Find students who had late submissions within the 48-hour penalty window

print("Examples of late penalty applications (lateness > 0 and < 48 hours):")
print("(Note: 'Individual Reflection' assignments are excluded from late penalties)\n")

# Check a few specific assignments for late penalties
for assignment in ['Linear Algebra', 'K-Means Clustering', 'Hierarchical Clustering and GMMs']:
    score_col = assignment
    lateness_col = f"{assignment} - Lateness (H:M:S)"
    
    if lateness_col in df.columns:
        # Find students with late submissions in penalty range
        for idx, row in df.iterrows():
            lateness_hours = parse_lateness_to_hours(row[lateness_col])
            if lateness_hours > 0 and lateness_hours < 48:
                original_score = row[score_col]
                adjusted_score = apply_late_penalty(original_score, lateness_hours)
                print(f"{row['Name']} - {assignment}:")
                print(f"  Original score: {original_score}, Lateness: {row[lateness_col]} ({lateness_hours:.2f}h)")
                print(f"  Adjusted score: {adjusted_score} (10% penalty applied)")
                print()
                break  # Just show one example per assignment


In [ ]:
# Summary statistics for the new score columns
print("Summary Statistics for New Score Columns:\n")
summary_cols = ['Total Homework Score', 'In-Class Activities Score', 'Midterm Challenge Score', 'Final Project Total Score']
print(df[summary_cols].describe())


In [ ]:
# Find students with late submissions past 48 hours
# (excluding 'Individual Reflection' assignments)
print("Students with late submissions past 48 hours:\n")
print("(Excluding 'Individual Reflection' and 'Personal Contributions' assignments)\n")

late_records = []

for assignment in assignment_columns:
    # Skip 'Individual Reflection' assignments
    if 'Individual Reflection' in assignment:
        continue
    if 'Personal Contributions' in assignment:
        continue
    
    lateness_col = f"{assignment} - Lateness (H:M:S)"
    score_col = assignment
    
    if lateness_col in df.columns:
        for idx, row in df.iterrows():
            lateness_hours = parse_lateness_to_hours(row[lateness_col])
            if lateness_hours >= 48:
                late_records.append({
                    'Name': row['Name'],
                    'Email': row['Email'],
                    'Assignment': assignment,
                    'Score': row[score_col],
                    'Max Points': row[f"{assignment} - Max Points"] if f"{assignment} - Max Points" in df.columns else None,
                    'Lateness': row[lateness_col],
                    'Lateness (hours)': round(lateness_hours, 2)
                })

# Create a dataframe of late submissions
late_df = pd.DataFrame(late_records)

if len(late_df) > 0:
    # Sort by student name then assignment
    late_df = late_df.sort_values(['Name', 'Assignment'])
    print(f"Total late submissions (>= 48 hours): {len(late_df)}")
    print(f"Number of unique students affected: {late_df['Name'].nunique()}")
    print()
    
    # Display all late submissions
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', 50)
    display(late_df)
else:
    print("No late submissions past 48 hours found.")


In [ ]:
# Write the entire dataframe to an Excel file
output_file = 'DS701_Fall_2025_grades_processed.xlsx'

df.to_excel(output_file, index=False, engine='openpyxl')

print(f"Dataframe written to: {output_file}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

